In [1]:
import pandas as pd
import torch
import subprocess
import platform
from transformers import pipeline, AutoTokenizer
from collections import Counter
from tqdm import tqdm


In [2]:
# Custom functions

from custom_functions import (
    get_device,
    get_pipeline_device_id,
    normalize_label,
    analyze_long_text,
    analyze_sentiment,
    clean_text,
)


In [3]:
toaster_df = pd.DataFrame({
    "RV_TRANS": [
        # Short reviews
        "Love making 4 slices at a time.",
        "Great item to use if you love grilled cheese because you can adjust the heat.",
        ".",        # invalid — skipped
        None,       # invalid — skipped
        "This oven works very well, and does everything I need it to do in the kitchen.",
        "Only toasts one side of the sandwich. Slots are too narrow. Very disappointed overall.",

        # Long reviews (will trigger chunking)
        """I have been using this toaster for about six months now and I have a lot to say about it. 
        When I first received it, I was immediately impressed by the build quality and the sleek design. 
        It looks fantastic on my kitchen counter and fits in well with my other appliances. Setting it up 
        was straightforward and the instructions were clear and easy to follow. In terms of performance, 
        the toaster heats up quickly and produces evenly toasted bread every single time. I have tried it 
        with regular sandwich bread, thick sourdough, bagels, and even frozen waffles, and it handles all 
        of them beautifully. The wide slots are a great feature that I did not know I needed until I started 
        using them. The browning settings are accurate and consistent, which is something I struggled with 
        on my previous toaster. The crumb tray is easy to remove and clean, which makes maintenance a breeze. 
        I also appreciate the cancel button, which works instantly without any lag. One minor issue I noticed 
        is that the exterior gets quite hot during extended use, so you need to be careful if you have young 
        children around. Overall, I am extremely happy with this purchase and would highly recommend it to 
        anyone looking for a reliable and stylish toaster for everyday use. Five stars without hesitation.""",
    ],
    "RV_DT": [
        "2019-01-06", "2018-12-09", "2021-03-28", "2022-11-17",
        "2020-10-04", "2020-10-04", "2022-10-03", 
    ],
    "SRVS": ["positive", "positive", "neutral", "neutral", "positive", "negative", "positive"],
})

In [4]:
toaster_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   RV_TRANS  6 non-null      str  
 1   RV_DT     7 non-null      str  
 2   SRVS      7 non-null      str  
dtypes: str(3)
memory usage: 300.0 bytes


In [5]:
# Summarize text length in RV_TRANS

text = toaster_df["RV_TRANS"].astype("string")

length_summary = pd.DataFrame({
    "characters": text.str.len(),
    "words": text.str.split().str.len()
})

length_summary.describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.95, 0.99])

,characters
count,6.0
mean,276.833333
std,545.361318
min,1.0
25%,42.5
50%,77.5
75%,84.0
90%,737.0
95%,1062.5
99%,1322.9


In [6]:
char_counts = text.str.len()
(char_counts > 512).mean()

np.float64(0.16666666666666666)

In [7]:
# ── 2. Device Detection ───────────────────────────────────────────────────────
device = get_device()
device_id = get_pipeline_device_id(device)


Using Apple MPS (Apple M2 Max)


In [ ]:

# ── 3. Model & Tokenizer ──────────────────────────────────────────────────────
MODEL = "cardiffnlp/twitter-roberta-base-sentiment-latest"

print(f"\nLoading model: {MODEL}")
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model=MODEL,
    tokenizer=MODEL,
    device=device_id,
    truncation=True,
    max_length=512,
    batch_size=32,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL)
print("Model loaded\n")



📦 Loading model: cardiffnlp/twitter-roberta-base-sentiment-latest


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded



In [9]:
#── 4. Label Mapping ──────────────────────────────────────────────────────────
LABEL_MAP = {
    "positive": "positive",
    "negative": "negative",
    "neutral":  "neutral",
    "label_0":  "negative",
    "label_1":  "neutral",
    "label_2":  "positive",
}


In [10]:
# ── 8. Run ────────────────────────────────────────────────────────────────────
toaster_df = analyze_sentiment(
    toaster_df,
    sentiment_pipeline=sentiment_pipeline,
    text_col="RV_TRANS",
    label_map=LABEL_MAP,
)

Analyzing 5 reviews (skipping 2 empty/invalid)...

Short reviews (direct batch): 5
Long reviews (chunking + voting): 0



Short texts:   0%|          | 0/1 [00:00<?, ?it/s]


Done. 0 review(s) used chunking + voting.


In [11]:
# ── 9. Results ────────────────────────────────────────────────────────────────
print("\n📊 Results:")
print(toaster_df[["RV_TRANS", "RV_DT", "SENTIMENT", "SENTIMENT_SCORE", "CHUNKED"]].to_string())

print("\n📈 Sentiment Distribution:")
print(toaster_df["SENTIMENT"].value_counts())

print("\n🔎 Long reviews that used chunking:")
print(toaster_df[toaster_df["CHUNKED"]][["RV_TRANS", "SENTIMENT", "SENTIMENT_SCORE"]])


📊 Results:
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            

In [13]:
# Compare SRVS and SENTIMENT
compare_df = toaster_df[["SRVS", "SENTIMENT", "RV_TRANS"]].copy()


In [14]:

# Normalize text labels
compare_df["SRVS_clean"] = (
    compare_df["SRVS"]
    .astype("string")
    .str.strip()
    .str.lower()
)

compare_df["SENTIMENT_clean"] = (
    compare_df["SENTIMENT"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Keep only rows where both columns are available
compare_valid = compare_df.dropna(subset=["SRVS_clean", "SENTIMENT_clean"]).copy()

# Check match
compare_valid["MATCH"] = (
    compare_valid["SRVS_clean"] == compare_valid["SENTIMENT_clean"]
)

# Agreement rate
agreement_rate = compare_valid["MATCH"].mean()

print(f"Agreement rate: {agreement_rate:.2%}")
print(f"Compared rows: {len(compare_valid)}")

# Cross-tab comparison
pd.crosstab(
    compare_valid["SRVS_clean"],
    compare_valid["SENTIMENT_clean"],
    margins=True
)


Agreement rate: 100.00%
Compared rows: 5


SENTIMENT_clean,negative,positive,All
SRVS_clean,,,
negative,1,0,1
positive,0,4,4
All,1,4,5
